# Charting Boulder: BVSD Retrieval
[Brian C. Keegan, Ph.D.](http://www.brianckeegan.com)
September 2026

Released under a [MIT License](https://opensource.org/licenses/MIT).

This notebook does the fetching and the tidying for the BVSD enrollment column. It reads the
sources in `data/raw/`, calls three keyed APIs, and writes tidy tables to `data/processed/`.
It draws no charts and makes no argument. The argument is in `bvsd-analysis.ipynb`, which reads
only `data/processed/` and never touches the network.

The split has one rule. This notebook may reshape, filter, join and check. It may not compute a
measure the column reports. Percent changes, ratios, projections and robustness tests all belong
in the analysis notebook, where a reader looking for the reasoning will find it.

**Keys.** Two of the sources need a free API key, read from the environment and never written
into this file:

| Variable | Source | Sign up |
|---|---|---|
| `CENSUS_API_KEY` | Census Bureau (decennial and ACS) | <https://api.census.gov/data/key_signup.html> |
| `FRED_API_KEY` | Federal Reserve Bank of St. Louis | <https://fred.stlouisfed.org/docs/api/api_key.html> |

**Run time.** About seven minutes end to end. The FRED section makes 52 calls, the SEDAC section
reads eighteen 4 MB workbooks, and the open enrollment section parses 31 PDFs.

### Environment
Beyond a standard Anaconda distribution this notebook needs `requests`, `pdfplumber`, `openpyxl`
and `tabulate`.

In [1]:
import numpy as np
import pandas as pd

import hashlib, io, json, os, re, subprocess, time
from collections import defaultdict
from pathlib import Path

import pdfplumber
import requests

pd.options.display.max_columns = 100

RAW = Path('data/raw')
PROC = Path('data/processed')
PROC.mkdir(parents=True, exist_ok=True)

RETRY_STATUS = {429, 500, 502, 503, 504}

def api_key(variable, signup_url):
    # Keys live in the environment. A missing key stops the notebook; nothing is substituted.
    value = os.getenv(variable)
    if not value:
        raise RuntimeError(f'Set the {variable} environment variable (free from {signup_url})')
    return value

def api_get(url, params, attempts=4):
    # These endpoints answer dozens of calls in a row and return a gateway error now and then.
    # Retry the transient statuses only; anything else, and the last attempt, still raises.
    for attempt in range(attempts):
        response = requests.get(url, params=params, timeout=60)
        if response.status_code in RETRY_STATUS and attempt < attempts - 1:
            time.sleep(2 ** attempt)
            continue
        response.raise_for_status()
        return response.json()

written_l = []   # every file this notebook writes, for the manifest at the end

def write_table(frame, relative_path, note=''):
    path = PROC / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    written_l.append({'file': str(path), 'rows': len(frame), 'cols': frame.shape[1], 'note': note})
    print(f'{str(path):<48} {len(frame):>7,} rows x {frame.shape[1]} cols   {note}')
    return frame

def write_raw(payload, relative_path, note=''):
    path = RAW / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=1), encoding='utf-8')
    written_l.append({'file': str(path), 'rows': np.nan, 'cols': np.nan, 'note': note})
    print(f'{str(path):<48} {path.stat().st_size:>7,} bytes   {note}')

print('setup ok')

setup ok


## The place registry

`data/raw/places.csv` is this piece's own copy of the
[similar-cities basket](https://github.com/brianckeegan/charting-boulder/blob/main/2025-06-population/similar-boulder.json),
plus Boulder County's ring towns. Tier 1 is Boulder and the five ring towns that share its county;
tier 2 is the 25 peer cities. Every FIPS code is a fixed-width string, so it is read and written as
text; read as a number, a leading zero disappears and the join silently fails. `state` is the
postal abbreviation, which is what the FRED income series identifiers need.

In [2]:
places_df = pd.read_csv(RAW / 'places.csv', dtype={'place_fips': str, 'county_fips': str})

# Integrity checks
assert places_df['name'].is_unique, 'duplicate place name'
assert places_df['place_fips'].is_unique, 'duplicate place FIPS'
assert places_df['place_fips'].str.fullmatch(r'\d{7}').all(), 'place FIPS is not 7 digits'
assert places_df['county_fips'].str.fullmatch(r'\d{5}').all(), 'county FIPS is not 5 digits'
assert places_df['state'].str.fullmatch(r'[A-Z]{2}').all(), 'state is not a postal abbreviation'
assert (places_df['place_fips'].str[:2] == places_df['county_fips'].str[:2]).all(), \
    'place and county sit in different states'
assert set(places_df['tier']) == {1, 2}, f"unexpected tiers: {sorted(set(places_df['tier']))}"
assert (places_df['tier'] == 1).sum() == 6, 'tier 1 is Boulder plus the five ring towns'
assert (places_df['tier'] == 2).sum() == 25, 'expected 25 peer cities'
assert 'Boulder' in set(places_df['name']), 'the subject city is missing'

print(f"{len(places_df)} places · {(places_df['tier'] == 1).sum()} in Boulder County · "
      f"{(places_df['tier'] == 2).sum()} peers · {places_df['county_fips'].nunique()} counties")
write_table(places_df, 'places.csv', 'the registry, validated')
places_df.head()

31 places · 6 in Boulder County · 25 peers · 26 counties
data/processed/places.csv                             31 rows x 6 cols   the registry, validated


,name,state,place_fips,county_fips,tier,note
0,Boulder,CO,0807850,08013,1,subject city
1,Longmont,CO,0845970,08013,1,Boulder County ring; St. Vrain Valley SD
2,Lafayette,CO,0841835,08013,1,Boulder County ring
3,Louisville,CO,0846355,08013,1,Boulder County ring
4,Superior,CO,0875640,08013,1,Boulder County ring


## Decennial age tables from the NHGIS extracts

Four [IPUMS NHGIS](https://www.nhgis.org/) extracts sit in `data/raw/nhgis/`, one per census:

| Year | Dataset | Table | Grain |
|---|---|---|---|
| 1990 | STF1 | NP11 | age, grouped, no sex split |
| 2000 | SF1 | NP012B, NP014C | sex by age grouped; sex by single year under 20 |
| 2010 | SF1 | PCT12 | sex by single year |
| 2020 | DHC | PCT12 | sex by single year |

The 1990 and 2000 tables do not report single years of age above 19, so the finest grain shared by
all four censuses is the 19-group scheme below. Sexes are summed. Every age variable in each source
table must land in exactly one group, and every closed group must be tiled exactly once, or the
cell stops. The age labels are read from the codebook each extract ships with, not typed here, so
a changed extract cannot quietly shift a boundary.

In [3]:
NHGIS = RAW / 'nhgis'
GROUPS = [(0, 4), (5, 9), (10, 14), (15, 17), (18, 19), (20, 24), (25, 29), (30, 34),
          (35, 39), (40, 44), (45, 49), (50, 54), (55, 59), (60, 64), (65, 69),
          (70, 74), (75, 79), (80, 84), (85, 999)]
SINGLE_MAX = 19          # single years of age available in every census from 2000 on
EXTRA_COUNTIES = {'08014', '08123'}   # Broomfield (BVSD) and Weld (Erie), beyond the registry

# year -> (file stem, prefix of the grouped table, prefix of the single-year-under-20 table)
SOURCES = {1990: ('nhgis0005_ds120_1990', 'ET3', None),
           2000: ('nhgis0004_ds146_2000', 'FMZ', 'FNG'),
           2010: ('nhgis0004_ds173_2010', 'IC3', 'IC3'),
           2020: ('nhgis0004_ds259_2020', 'VCG', 'VCG')}

SEX_PREFIX = re.compile(r'^(?:Male|Female)\s*(?:>>|:)\s*')

def label_range(label):
    # 'Male: 7 to 9 years' -> (7, 9); 'Under 5 years' -> (0, 4); '85 years and over' -> (85, 999).
    # Returns None for the subtotal rows.
    text = SEX_PREFIX.sub('', label.strip())
    if text in ('Total', 'Male', 'Female'):
        return None
    if match := re.match(r'^Under (\d+) years?$', text):
        return 0, int(match.group(1)) - 1
    if match := re.match(r'^(\d+) years? and over$', text):
        return int(match.group(1)), 999
    if match := re.match(r'^(\d+) (?:to|and) (\d+) years?$', text):
        return int(match.group(1)), int(match.group(2))
    if match := re.match(r'^(\d+) years?$', text):
        return int(match.group(1)), int(match.group(1))
    raise ValueError(f'unparsed age label: {label!r}')

def codebook_vars(stem, geog, prefix):
    # {variable: (lo, hi)} for every age variable of one table, from the shipped codebook
    text = (NHGIS / f'{stem}_{geog}_codebook.txt').read_text(encoding='latin-1')
    found = {}
    for var, label in re.findall(rf'^\s+({prefix}\d{{3}}):\s+(.+?)\s*$', text, flags=re.M):
        age_range = label_range(label)
        if age_range is not None:
            found[var] = age_range
    assert found, f'no {prefix} variables found in {stem}_{geog} codebook'
    return found

def group_of(age_range):
    lo, hi = age_range
    hits = [group for group in GROUPS if lo >= group[0] and hi <= group[1]]
    assert len(hits) == 1, f'age range {age_range} does not fit exactly one group'
    return hits[0]

def check_coverage(vars_d):
    # Every closed group must be tiled exactly once by the ranges assigned to it
    for group in GROUPS:
        ranges = {r for r in vars_d.values() if group_of(r) == group}
        if group[1] == 999:
            assert ranges, f'group {group} has no source variable'
            continue
        width = sum(hi - lo + 1 for lo, hi in ranges)
        assert width == group[1] - group[0] + 1, f'group {group} covered by {sorted(ranges)}'

def read_extract(year, geog, prefix, keep_ids):
    stem = SOURCES[year][0]
    vars_d = codebook_vars(stem, geog, prefix)
    id_cols = ['STATEA', 'PLACEA'] if geog == 'place' else ['STATEA', 'COUNTYA']
    frame = pd.read_csv(NHGIS / f'{stem}_{geog}.csv', dtype=str, encoding='latin-1',
                        usecols=id_cols + list(vars_d))
    width = 5 if geog == 'place' else 3
    frame['fips'] = frame['STATEA'].str.zfill(2) + frame[id_cols[1]].str.zfill(width)
    frame = frame[frame['fips'].isin(keep_ids)].drop(columns=id_cols)
    frame[list(vars_d)] = frame[list(vars_d)].apply(pd.to_numeric)
    return frame, vars_d

def grouped_long(year, geog, keep_ids):
    frame, vars_d = read_extract(year, geog, SOURCES[year][1], keep_ids)
    check_coverage(vars_d)
    long_df = frame.melt(id_vars='fips', var_name='var', value_name='pop')
    long_df[['age_lo', 'age_hi']] = pd.DataFrame([group_of(vars_d[v]) for v in long_df['var']],
                                                 index=long_df.index)
    return (long_df.groupby(['fips', 'age_lo', 'age_hi'], as_index=False)['pop'].sum()
            .assign(year=year))

def single_long(year, keep_ids):
    frame, vars_d = read_extract(year, 'place', SOURCES[year][2], keep_ids)
    keep_d = {v: r[0] for v, r in vars_d.items() if r[0] == r[1] and r[0] <= SINGLE_MAX}
    assert sorted(set(keep_d.values())) == list(range(SINGLE_MAX + 1)), f'{year}: single ages incomplete'
    long_df = frame.melt(id_vars='fips', value_vars=list(keep_d), var_name='var', value_name='pop')
    long_df['age'] = long_df['var'].map(keep_d)
    return long_df.groupby(['fips', 'age'], as_index=False)['pop'].sum().assign(year=year)

def finish(frames, id_name):
    out_df = pd.concat(frames, ignore_index=True).rename(columns={'fips': id_name})
    out_df['age_group'] = out_df.apply(
        lambda row: f'{row.age_lo}+' if row.age_hi == 999 else f'{row.age_lo}-{row.age_hi}', axis=1)
    out_df['pop'] = out_df['pop'].astype(int)
    cols = [id_name, 'year', 'age_group', 'age_lo', 'age_hi', 'pop']
    return out_df[cols].sort_values([id_name, 'year', 'age_lo']).reset_index(drop=True)

print(f'{len(GROUPS)} age groups · {len(SOURCES)} censuses')

19 age groups · 4 censuses


In [4]:
place_ids = set(places_df['place_fips'])
county_ids = set(places_df['county_fips']) | EXTRA_COUNTIES

place_age_groups_df = finish([grouped_long(y, 'place', place_ids) for y in SOURCES], 'place_fips')
county_age_groups_df = finish([grouped_long(y, 'county', county_ids) for y in SOURCES], 'county_fips')
place_age_single_df = (pd.concat([single_long(y, place_ids) for y in (2000, 2010, 2020)],
                                 ignore_index=True)
                       .rename(columns={'fips': 'place_fips'})[['place_fips', 'year', 'age', 'pop']]
                       .astype({'pop': int}).sort_values(['place_fips', 'year', 'age'])
                       .reset_index(drop=True))

# Integrity checks: the single-year table must reproduce the grouped table where they overlap
single_check_s = (place_age_single_df.assign(age_lo=(place_age_single_df['age'] // 5) * 5)
                  .query('age_lo <= 5').groupby(['place_fips', 'year', 'age_lo'])['pop'].sum()
                  .rename('single'))
grouped_check_s = county_age_groups_df.set_index(['county_fips', 'year', 'age_lo'])['pop']
grouped_place_s = place_age_groups_df.set_index(['place_fips', 'year', 'age_lo'])['pop']
joined_df = pd.concat([single_check_s, grouped_place_s], axis=1, join='inner')
assert (joined_df['single'] == joined_df['pop']).all(), 'single-year and grouped tables disagree'

missing = place_ids - set(place_age_groups_df['place_fips'])
assert not missing, f'registry places absent from NHGIS: {sorted(missing)}'
per_year_s = place_age_groups_df.groupby('year')['place_fips'].nunique()
assert (per_year_s == len(place_ids)).all(), f'places per year: {per_year_s.to_dict()}'
assert place_age_groups_df['pop'].ge(0).all(), 'negative population'

write_table(place_age_groups_df, 'place-age-groups.csv',
            f"{place_age_groups_df.place_fips.nunique()} places x {sorted(SOURCES)}")
write_table(county_age_groups_df, 'county-age-groups.csv',
            f"{county_age_groups_df.county_fips.nunique()} counties (Broomfield exists from 2010)")
write_table(place_age_single_df, 'place-age-single.csv',
            f"ages 0-{SINGLE_MAX} · {sorted(place_age_single_df.year.unique().tolist())}")
place_age_groups_df.head()

data/processed/place-age-groups.csv                2,356 rows x 6 cols   31 places x [1990, 2000, 2010, 2020]
data/processed/county-age-groups.csv               2,090 rows x 6 cols   28 counties (Broomfield exists from 2010)
data/processed/place-age-single.csv                1,860 rows x 4 cols   ages 0-19 · [2000, 2010, 2020]


,place_fips,year,age_group,age_lo,age_hi,pop
0,0137000,1990,0-4,0,4,10562
1,0137000,1990,5-9,5,9,10516
2,0137000,1990,10-14,10,14,9893
3,0137000,1990,15-17,15,17,6076
4,0137000,1990,18-19,18,19,5810


## State Demography Office forecasts

The [Colorado State Demography Office](https://demography.dola.colorado.gov/assets/html/sdodata.html)
publishes the official Vintage 2024 county forecasts. Single-year-of-age population and components
of change run to 2060; the household projection stops at 2050 and is not extended anywhere in this
piece. BVSD spans two counties, Boulder (SDO code 13) and Broomfield (14). The first row of each
file is a vintage title line, so each read skips it.

The components and household tables are cut to the two BVSD counties. The single-year-age table is
kept for the seventeen Colorado counties of the
[Front Range Urban Corridor](https://en.wikipedia.org/wiki/Front_Range_Urban_Corridor) — El Paso,
Denver, Arapahoe, Jefferson, Adams, Larimer, Douglas, Boulder, Weld, Pueblo, Broomfield, Fremont,
Elbert, Teller, Park, Clear Creek and Gilpin — so the analysis can set Boulder County against the
region it sits in. The file also publishes a Colorado row under county code 0; it is kept, and
checked against the sum of all sixty-four counties rather than trusted.

A row count that lands exactly on an Excel sheet limit is treated as a truncated download, not as
data.

In [5]:
SDO = RAW / 'sdo'
BVSD_COUNTIES = [13, 14]

sya_df = pd.read_csv(SDO / 'sya-county.csv', skiprows=1)
components_df = pd.read_excel(SDO / 'components-change-county.xlsx', skiprows=1)
households_df = pd.read_excel(SDO / 'household-county.xlsx', skiprows=1)

# Integrity checks
for name, frame, key in [('sya', sya_df, 'countyfips'), ('components', components_df, 'countyfips'),
                         ('households', households_df, 'area_code')]:
    frame[key] = frame[key].astype(int)
    assert set(BVSD_COUNTIES) <= set(frame[key]), f'{name}: missing a BVSD county'
    assert len(frame) not in (256, 16_384, 65_536, 1_048_576), f'{name}: possible Excel truncation'
assert sya_df['age'].between(0, 100).all(), 'sya: age outside 0-100'
assert sya_df['totalpopulation'].notna().all(), 'sya: null population'
assert sya_df['year'].between(1990, 2060).all() and components_df['year'].between(1970, 2060).all()

# The seventeen Colorado counties of the Front Range Urban Corridor, plus the published state row
FRONT_RANGE_COUNTIES = ['Adams', 'Arapahoe', 'Boulder', 'Broomfield', 'Clear Creek', 'Denver',
                        'Douglas', 'El Paso', 'Elbert', 'Fremont', 'Gilpin', 'Jefferson',
                        'Larimer', 'Park', 'Pueblo', 'Teller', 'Weld']
COLORADO_CODE = 0
assert set(FRONT_RANGE_COUNTIES) <= set(sya_df['county']), 'a Front Range county is not in the file'

# The state row must be the sum of the counties, or one of the two is not what it claims. The SDO
# rounds each county independently, so the two sides part by a few dozen people in 6.7 million;
# the check is on the relative gap, not on an exact match.
county_sum_s = sya_df.loc[sya_df['countyfips'] != COLORADO_CODE].groupby('year')['totalpopulation'].sum()
state_row_s = sya_df.loc[sya_df['countyfips'] == COLORADO_CODE].groupby('year')['totalpopulation'].sum()
state_gap_s = (county_sum_s - state_row_s).abs() / state_row_s
assert state_gap_s.max() < 1e-4, f'the Colorado row is not the sum of its counties: {state_gap_s.max():.2%}'

c0 = sya_df['county'].isin(FRONT_RANGE_COUNTIES) | (sya_df['countyfips'] == COLORADO_CODE)
c1 = components_df['countyfips'].isin(BVSD_COUNTIES)
c2 = households_df['area_code'].isin(BVSD_COUNTIES)

sdo_sya_df = (sya_df.loc[c0, ['countyfips', 'county', 'year', 'age', 'totalpopulation']]
              .sort_values(['countyfips', 'year', 'age']).reset_index(drop=True))
sdo_components_df = (components_df.loc[c1, ['countyfips', 'year', 'births', 'deaths', 'netmig']]
                     .sort_values(['countyfips', 'year']).reset_index(drop=True))
sdo_households_df = (households_df.loc[c2, ['area_code', 'year', 'age_group_id',
                                            'household_type_id', 'total_households']]
                     .sort_values(['area_code', 'year', 'household_type_id']).reset_index(drop=True))

# Every county-year must carry an unbroken age ladder from 0, or a cohort sum silently
# short-changes. The top age is 90 in the estimate years and 100 from 2010 on.
ladder_df = sdo_sya_df.groupby(['countyfips', 'year'])['age'].agg(['min', 'max', 'nunique'])
assert sdo_sya_df['county'].nunique() == len(FRONT_RANGE_COUNTIES) + 1, 'an area went missing'
assert (ladder_df['min'] == 0).all(), 'sya: a county-year does not start at age 0'
assert (ladder_df['nunique'] == ladder_df['max'] + 1).all(), 'sya: a county-year skips an age'
assert set(ladder_df['max']) <= {90, 100}, f"sya: unexpected top age {sorted(set(ladder_df['max']))}"

print(f'Colorado row matches the sum of {sya_df.countyfips.nunique() - 1} counties to within '
      f'{state_gap_s.max() * 100:.5f}% (SDO rounds each county on its own)')
print(f'single-year age {sdo_sya_df.year.min()}-{sdo_sya_df.year.max()} · '
      f'components {sdo_components_df.year.min()}-{sdo_components_df.year.max()} · '
      f'households {sdo_households_df.year.min()}-{sdo_households_df.year.max()}')
print('vintage note: single-year age is the Mar 2025 prep; components and households are Mar 2026. '
      'All Vintage 2024.')
write_table(sdo_sya_df, 'sdo-sya-frontrange.csv',
            f'{len(FRONT_RANGE_COUNTIES)} Front Range counties + the Colorado row')
write_table(sdo_components_df, 'sdo-components-bvsd.csv', 'births, deaths, net migration')
write_table(sdo_households_df, 'sdo-households-bvsd.csv', 'households by type and householder age')
sdo_sya_df.head()

Colorado row matches the sum of 64 counties to within 0.00058% (SDO rounds each county on its own)
single-year age 1990-2060 · components 1970-2060 · households 2010-2050
vintage note: single-year age is the Mar 2025 prep; components and households are Mar 2026. All Vintage 2024.
data/processed/sdo-sya-frontrange.csv            125,478 rows x 5 cols   17 Front Range counties + the Colorado row
data/processed/sdo-components-bvsd.csv               152 rows x 5 cols   births, deaths, net migration
data/processed/sdo-households-bvsd.csv             2,050 rows x 5 cols   households by type and householder age


,countyfips,county,year,age,totalpopulation
0,0,Colorado,1990,0,52545
1,0,Colorado,1990,1,51005
2,0,Colorado,1990,2,50601
3,0,Colorado,1990,3,51361
4,0,Colorado,1990,4,52210


## County projections to 2100 from the SSPs

The Colorado State Demography Office forecasts Colorado. It says nothing about the peer counties,
so on its own it cannot answer whether the future the district is planning for is unusual. The
[SEDAC county projections](https://sedac.ciesin.columbia.edu/data/set/popdynamics-us-county-level-pop-projections-sex-race-age-ssp-2020-2100)
(Hauer 2019, published by CIESIN in 2021) do cover every county, in five-year steps from 2020 to
2100, under the five Shared Socioeconomic Pathways.

Three properties of this source shape what can be built from it:

- **Counties only.** There is no place grain, so nothing here can be compared with a city figure.
- **Eighteen five-year age groups**, 0-4 through 85+. The 15-19 group is atomic, so 15-17 and 18-19
  cannot be separated and no bucket may cut between 17 and 18.
- **The 2020 column is a projection**, not the 2020 census count. Its inputs predate the census.
  The analysis notebook reports how far each county's 2020 projection sits from its census count
  and computes every rate of change inside this source, never across the two.

Hauer's own documentation warns that a small early error compounds over a long horizon, and both
Striessnig et al. (2019) and Jiang et al. (2020) hold that the method suits shorter periods than
2100. The tidy table keeps the whole published series; the column reads only as far as 2050.

The 25 workbooks ship one file per series. This notebook reads the eighteen age files, keeps the
26 basket counties plus Broomfield, and sums every county in each file for a national total. About
a minute.

In [6]:
SEDAC = RAW / 'sedac'
# SEDAC age group -> (file stem, age_lo, age_hi). Eighteen groups, tiling 0 to 85+.
SEDAC_AGE_GROUPS = {
    '0-4': ('0_4', 0, 4), '5-9': ('5_9', 5, 9), '10-14': ('10_14', 10, 14),
    '15-19': ('15_19', 15, 19), '20-24': ('20_24', 20, 24), '25-29': ('25_29', 25, 29),
    '30-34': ('30_34', 30, 34), '35-39': ('35_39', 35, 39), '40-44': ('40_44', 40, 44),
    '45-49': ('45_49', 45, 49), '50-54': ('50_54', 50, 54), '55-59': ('55_59', 55, 59),
    '60-64': ('60_64', 60, 64), '65-69': ('65_69', 65, 69), '70-74': ('70_74', 70, 74),
    '75-79': ('75_79', 75, 79), '80-84': ('80_84', 80, 84), '85+': ('85_over', 85, 999)}
SSP_SCENARIOS = [1, 2, 3, 4, 5]
SEDAC_YEARS = list(range(2020, 2105, 5))
SEDAC_VALUE_COLS = [f'ssp{ssp}{year}' for year in SEDAC_YEARS for ssp in SSP_SCENARIOS]

# Boulder County, the 25 peer counties, and Broomfield, which BVSD also spans
BVSD_COUNTY_FIPS = {'08013', '08014'}
basket_county_s = places_df.loc[(places_df['tier'] == 2) | (places_df['name'] == 'Boulder'),
                                'county_fips']
SEDAC_KEEP = set(basket_county_s) | BVSD_COUNTY_FIPS

def read_sedac_age_file(label):
    stem, age_lo, age_hi = SEDAC_AGE_GROUPS[label]
    frame = pd.read_excel(SEDAC / f'hauer_county_{stem}_pop_SSPs.xlsx',
                          usecols=['GEOID10', 'geoid'] + SEDAC_VALUE_COLS)
    # 86 rows carry a null geoid and all-zero values: Puerto Rico, three Alaska areas and a
    # handful the 2010 shapefile could not match. They are dropped, not zero-filled.
    unmatched_df = frame[frame['geoid'].isna()]
    assert (unmatched_df[SEDAC_VALUE_COLS].to_numpy() == 0).all(), f'{label}: unmatched row is not empty'
    frame = frame[frame['geoid'].notna()].copy()
    frame['county_fips'] = frame['GEOID10'].astype(int).astype(str).str.zfill(5)
    assert frame['county_fips'].is_unique, f'{label}: duplicate county'
    long_df = frame.melt(id_vars='county_fips', value_vars=SEDAC_VALUE_COLS,
                         var_name='var', value_name='pop')
    long_df['ssp'] = long_df['var'].str[3].astype(int)
    long_df['year'] = long_df['var'].str[4:].astype(int)
    long_df = long_df.assign(age_group=label, age_lo=age_lo, age_hi=age_hi)
    return long_df, len(unmatched_df), frame['county_fips'].nunique()

sedac_county_l, sedac_national_l, unmatched_n, county_n = [], [], set(), set()
for label in SEDAC_AGE_GROUPS:
    long_df, n_unmatched, n_counties = read_sedac_age_file(label)
    unmatched_n.add(n_unmatched); county_n.add(n_counties)
    sedac_national_l.append(long_df.groupby(['age_group', 'age_lo', 'age_hi', 'ssp', 'year'],
                                            as_index=False)['pop'].sum())
    sedac_county_l.append(long_df.loc[long_df['county_fips'].isin(SEDAC_KEEP),
                                      ['county_fips', 'age_group', 'age_lo', 'age_hi',
                                       'ssp', 'year', 'pop']])

sedac_county_df = pd.concat(sedac_county_l, ignore_index=True)
sedac_national_df = pd.concat(sedac_national_l, ignore_index=True)
for frame in (sedac_county_df, sedac_national_df):
    frame.sort_values(['ssp', 'year', 'age_lo'], inplace=True, kind='stable')
    frame.reset_index(drop=True, inplace=True)

# Integrity checks
expected_cells = len(SEDAC_AGE_GROUPS) * len(SSP_SCENARIOS) * len(SEDAC_YEARS)
assert len(county_n) == 1 and len(unmatched_n) == 1, 'the age files disagree on their county list'
assert set(sedac_county_df['county_fips']) == SEDAC_KEEP, 'a basket county is missing'
assert len(sedac_county_df) == len(SEDAC_KEEP) * expected_cells, 'county row count is wrong'
assert len(sedac_national_df) == expected_cells, 'national row count is wrong'
for name, frame in [('county', sedac_county_df), ('national', sedac_national_df)]:
    assert frame['pop'].gt(0).all(), f'{name}: non-positive population'
    assert set(frame['ssp']) == set(SSP_SCENARIOS), f'{name}: missing a scenario'
    assert set(frame['year']) == set(SEDAC_YEARS), f'{name}: missing a year'
# The eighteen groups must tile without a gap or an overlap
edges_l = sorted((lo, hi) for _, lo, hi in SEDAC_AGE_GROUPS.values())
assert edges_l[0][0] == 0 and all(b[0] == a[1] + 1 for a, b in zip(edges_l, edges_l[1:])),     'the SEDAC age groups do not tile'

print(f'{county_n.pop():,} counties in each file, {unmatched_n.pop()} unmatched rows dropped')
print(f'{len(SEDAC_AGE_GROUPS)} age groups x {len(SSP_SCENARIOS)} scenarios x '
      f'{len(SEDAC_YEARS)} years, {SEDAC_YEARS[0]}-{SEDAC_YEARS[-1]}')
print(f'kept {len(SEDAC_KEEP)} counties: the 26-county basket plus Broomfield')
write_table(sedac_county_df, 'sedac-county-age-ssp.csv', 'basket counties, all five SSPs')
write_table(sedac_national_df, 'sedac-national-age-ssp.csv', 'every US county summed')
sedac_county_df.head()

3,135 counties in each file, 86 unmatched rows dropped
18 age groups x 5 scenarios x 17 years, 2020-2100
kept 27 counties: the 26-county basket plus Broomfield
data/processed/sedac-county-age-ssp.csv           41,310 rows x 7 cols   basket counties, all five SSPs
data/processed/sedac-national-age-ssp.csv          1,530 rows x 6 cols   every US county summed


,county_fips,age_group,age_lo,age_hi,ssp,year,pop
0,06059,0-4,0,4,1,2020,205701.713135
1,06083,0-4,0,4,1,2020,30168.059387
2,42003,0-4,0,4,1,2020,68238.139038
3,37063,0-4,0,4,1,2020,23378.867554
4,48041,0-4,0,4,1,2020,18814.182678


## National age counts from the Census Bureau API

The peer basket holds 25 college towns, which share a shape no national figure has. A gap against
the peer median and a gap against the country therefore say different things, so the analysis
carries both. The national counts come from the same two censuses through the
[Census Bureau API](https://www.census.gov/data/developers/data-sets.html), table P12, whose rows
cut at 15-17 and 18-19 and so support the same age buckets the places use.

P12 numbers rows 3 to 25 as men by age and 27 to 49 as the same ages for women. The 2010 file names
them `P012003`; the 2020 file names them `P12_003N`. Both answers are saved to `data/raw/census/`
as returned.

In [7]:
DECENNIAL_TABLE = {2010: ('https://api.census.gov/data/2010/dec/sf1', 'P012{:03d}', 'P012001'),
                   2020: ('https://api.census.gov/data/2020/dec/dhc', 'P12_{:03d}N', 'P12_001N')}
P12_AGE_ROWS = [(0, 4), (5, 9), (10, 14), (15, 17), (18, 19), (20, 20), (21, 21), (22, 24),
                (25, 29), (30, 34), (35, 39), (40, 44), (45, 49), (50, 54), (55, 59), (60, 61),
                (62, 64), (65, 66), (67, 69), (70, 74), (75, 79), (80, 84), (85, 999)]

CENSUS_KEY = api_key('CENSUS_API_KEY', 'https://api.census.gov/data/key_signup.html')

def national_age_groups(year):
    url, var_fmt, total_var = DECENNIAL_TABLE[year]
    male_vars = [var_fmt.format(3 + i) for i in range(len(P12_AGE_ROWS))]
    female_vars = [var_fmt.format(27 + i) for i in range(len(P12_AGE_ROWS))]
    payload = api_get(url, {'get': ','.join([total_var] + male_vars + female_vars),
                            'for': 'us:1', 'key': CENSUS_KEY})
    write_raw(payload, f'census/national-p12-{year}.json', f'table P12, {year}, as returned')
    cell_d = dict(zip(payload[0], payload[1]))
    groups_df = pd.DataFrame([{'year': year, 'age_lo': lo, 'age_hi': hi,
                               'pop': int(cell_d[male_vars[i]]) + int(cell_d[female_vars[i]])}
                              for i, (lo, hi) in enumerate(P12_AGE_ROWS)])
    # The sexes and the age rows must add back to the published national total
    assert groups_df['pop'].sum() == int(cell_d[total_var]), f'{year}: P12 rows do not sum'
    return groups_df

national_age_groups_df = pd.concat([national_age_groups(year) for year in DECENNIAL_TABLE],
                                   ignore_index=True)
national_age_groups_df['age_group'] = national_age_groups_df.apply(
    lambda row: f'{row.age_lo}+' if row.age_hi == 999 else f'{row.age_lo}-{row.age_hi}', axis=1)
national_age_groups_df = national_age_groups_df[['year', 'age_group', 'age_lo', 'age_hi', 'pop']]

assert len(national_age_groups_df) == 2 * len(P12_AGE_ROWS), 'wrong row count'
for year in DECENNIAL_TABLE:
    total = national_age_groups_df.loc[national_age_groups_df['year'] == year, 'pop'].sum()
    print(f'  {year}: {total:,} people across {len(P12_AGE_ROWS)} age rows')
write_table(national_age_groups_df, 'national-age-groups.csv', 'decennial P12, both censuses')
national_age_groups_df.head()

data/raw/census/national-p12-2010.json             1,268 bytes   table P12, 2010, as returned


data/raw/census/national-p12-2020.json             1,316 bytes   table P12, 2020, as returned
  2010: 308,745,538 people across 23 age rows
  2020: 331,449,281 people across 23 age rows
data/processed/national-age-groups.csv                46 rows x 5 cols   decennial P12, both censuses


,year,age_group,age_lo,age_hi,pop
0,2010,0-4,0,4,20201362
1,2010,5-9,5,9,20348657
2,2010,10-14,10,14,20677194
3,2010,15-17,15,17,12954254
4,2010,18-19,18,19,9086089


## The annual survey: under-5 estimates

The decennial counts are ten years apart. The American Community Survey publishes an under-5
estimate every year, so the analysis asks whether that annual series can referee the decennial
finding. It needs the margin of error as much as the estimate, so both are pulled.

`B01001_003` is men under 5 and `B01001_027` women under 5; the `M` variables are the 90 percent
margins. The two margins are independent, so they combine in quadrature. One-year estimates cover
places of 65,000 people or more, which is why only four of the basket's cities appear here. The
Census Bureau released no standard one-year estimates for 2020.

In [8]:
ACS_URL = 'https://api.census.gov/data/{year}/acs/acs1'
ACS_YEARS = [year for year in range(2008, 2025) if year != 2020]
ACS_PLACES = {'Boulder': ('08', '07850'), 'Fort Collins': ('08', '27425'),
              'Madison': ('55', '48000'), 'Ann Arbor': ('26', '03000')}

acs_raw_d, acs_records_l = {}, []
for city, (state_fips, place_fips) in ACS_PLACES.items():
    for year in ACS_YEARS:
        payload = api_get(ACS_URL.format(year=year),
                          {'get': 'B01001_003E,B01001_003M,B01001_027E,B01001_027M',
                           'for': f'place:{place_fips}', 'in': f'state:{state_fips}',
                           'key': CENSUS_KEY})
        acs_raw_d[f'{city}|{year}'] = payload
        male_est, male_moe, female_est, female_moe = [float(v) for v in payload[1][:4]]
        acs_records_l.append({'city': city, 'year': year, 'under_5': male_est + female_est,
                              'moe': float(np.hypot(male_moe, female_moe))})

acs_under5_df = pd.DataFrame(acs_records_l)
acs_under5_df['moe_pct'] = acs_under5_df['moe'] / acs_under5_df['under_5'] * 100

# Integrity checks
assert len(acs_under5_df) == len(ACS_PLACES) * len(ACS_YEARS), 'a city-year is missing'
assert acs_under5_df['under_5'].gt(0).all(), 'non-positive estimate'
assert acs_under5_df['moe'].gt(0).all(), 'non-positive margin'
assert 2020 not in set(acs_under5_df['year']), '2020 has no standard one-year release'

write_raw(acs_raw_d, 'census/acs1-under5.json', 'one-year B01001, as returned')
print(f'{len(ACS_PLACES)} cities x {len(ACS_YEARS)} years '
      f'({ACS_YEARS[0]}-{ACS_YEARS[-1]}, no 2020)')
write_table(acs_under5_df, 'acs-under5.csv', 'estimate and 90% margin of error')
acs_under5_df.head()

data/raw/census/acs1-under5.json                  12,742 bytes   one-year B01001, as returned
4 cities x 16 years (2008-2024, no 2020)
data/processed/acs-under5.csv                         64 rows x 5 cols   estimate and 90% margin of error


,city,year,under_5,moe,moe_pct
0,Boulder,2008,4990.0,1250.971223,25.069564
1,Boulder,2009,3905.0,905.925494,23.199116
2,Boulder,2010,4274.0,937.057629,21.924605
3,Boulder,2011,4461.0,1009.311151,22.625222
4,Boulder,2012,3942.0,856.304852,21.722599


## House prices and incomes from FRED

Prices are the [FHFA All-Transactions House Price Index](https://www.fhfa.gov/data/hpi) at county
grain, and incomes are the Census Bureau's
[Small Area Income and Poverty Estimates](https://www.census.gov/programs-surveys/saipe.html) of
median household income, both read from [FRED](https://fred.stlouisfed.org/). The 26 counties are
Boulder plus the 25 peers; the ring towns share Boulder's county and would only repeat it.

Both series are annual after a mean over the observations FRED returns. Two gaps to know about.
The income series publishes no value for 1990, 1991, 1992, 1994 or 1996, in every county alike.
And New Haven County's income stops in 2021, when Connecticut replaced counties with planning
regions for federal statistics. Neither gap is filled here.

This cell makes 52 calls. Expect about a minute.

In [9]:
FRED_URL = 'https://api.stlouisfed.org/fred/series/observations'
FRED_KEY = api_key('FRED_API_KEY', 'https://fred.stlouisfed.org/docs/api/api_key.html')

def fred_observations(series_id):
    return api_get(FRED_URL, {'series_id': series_id, 'api_key': FRED_KEY,
                              'file_type': 'json'})['observations']

def annual_records(observations, city, series_id):
    frame = pd.DataFrame(observations)
    frame['value'] = pd.to_numeric(frame['value'], errors='coerce')   # FRED encodes missing as '.'
    frame['year'] = pd.to_datetime(frame['date']).dt.year
    annual_s = frame.dropna(subset=['value']).groupby('year')['value'].mean()
    return [{'city': city, 'series_id': series_id, 'year': int(y), 'value': float(v)}
            for y, v in annual_s.items()]

c0 = (places_df['tier'] == 2) | (places_df['name'] == 'Boulder')
housing_places_df = places_df.loc[c0].set_index('name')
assert len(housing_places_df) == 26, f'expected 26 cities; got {len(housing_places_df)}'

fred_raw_d, hpi_records_l, income_records_l = {}, [], []
for city, row in housing_places_df.iterrows():
    for series_id, sink in [(f"ATNHPIUS{row['county_fips']}A", hpi_records_l),
                            (f"MHI{row['state']}{row['county_fips']}A052NCEN", income_records_l)]:
        observations = fred_observations(series_id)
        fred_raw_d[series_id] = observations
        sink.extend(annual_records(observations, city, series_id))

county_hpi_df = pd.DataFrame(hpi_records_l)
county_income_df = pd.DataFrame(income_records_l)

# Integrity checks
for label, frame in [('house price index', county_hpi_df), ('median income', county_income_df)]:
    assert set(frame['city']) == set(housing_places_df.index), f'{label}: cities do not match'
    assert frame['value'].gt(0).all(), f'non-positive {label}'
    assert not frame.duplicated(['city', 'year']).any(), f'{label}: duplicate city-year'
assert county_income_df['year'].min() == 1989 and county_hpi_df['year'].min() <= 1975
income_gap_l = sorted(set(range(1989, 2025)) - set(county_income_df['year']))
assert income_gap_l == [1990, 1991, 1992, 1994, 1996], f'unexpected income gaps: {income_gap_l}'

write_raw(fred_raw_d, 'fred/observations.json', f'{len(fred_raw_d)} series, as returned')
print(f"{len(housing_places_df)} counties · house price index {county_hpi_df.year.min()}-"
      f"{county_hpi_df.year.max()} · median income {county_income_df.year.min()}-"
      f"{county_income_df.year.max()}")
print(f'income publishes no value for {income_gap_l} in every county')
print('income series ending before 2024:',
      {city: int(frame['year'].max())
       for city, frame in county_income_df.groupby('city') if frame['year'].max() < 2024})
write_table(county_hpi_df, 'county-hpi.csv', 'FHFA all-transactions index, annual mean')
write_table(county_income_df, 'county-income.csv', 'SAIPE median household income, annual mean')
county_hpi_df.head()

data/raw/fred/observations.json                  275,887 bytes   52 series, as returned
26 counties · house price index 1975-2025 · median income 1989-2024
income publishes no value for [1990, 1991, 1992, 1994, 1996] in every county
income series ending before 2024: {'New Haven': 2021}
data/processed/county-hpi.csv                      1,317 rows x 4 cols   FHFA all-transactions index, annual mean
data/processed/county-income.csv                     803 rows x 4 cols   SAIPE median household income, annual mean


,city,series_id,year,value
0,Boulder,ATNHPIUS08013A,1975,16.29
1,Boulder,ATNHPIUS08013A,1976,18.03
2,Boulder,ATNHPIUS08013A,1977,21.45
3,Boulder,ATNHPIUS08013A,1978,25.36
4,Boulder,ATNHPIUS08013A,1979,29.50


## Building permits, by place and by county

Two sources, because neither answers the question alone.

The [Census Building Permits Survey](https://www.census.gov/construction/bps/) publishes one
annual file per region listing every permit-issuing jurisdiction, with housing units split by
structure size: 1-unit, 2-unit, 3-4 unit and 5-or-more. Its FIPS place codes are the same codes the
registry uses, so the 25 peer cities and the Boulder County ring towns line up directly. This is
the only source in this piece that reaches the City of Boulder itself rather than its county.

FRED carries `BPPRIV` for every county, which is the county total with no structure split. The
title says "Structures" and the values are units: the sixty-four Colorado county series sum to the
Colorado "Units" series within a few percent. It is here as context, because the gap between the
city and its county is a finding in its own right.

Two coverage facts the analysis has to work around, both found by checking rather than assumed:

- **Bloomington, Indiana** files no separate place record. Monroe County reports as a single
  jurisdiction, so the city cannot be separated and appears only in the county table.
- **New Haven** is the reverse. The city files a full sixteen years, while FRED's New Haven County
  series ends in 2022 and its population series in 2021, both discontinued when Connecticut
  replaced counties with planning regions for federal statistics.

The region files run about 45 MB across sixteen years, so the raw copy kept here is the rows for
the registry's places, with every column as the file supplies it, rather than the whole download.

In [10]:
BPS_URL = 'https://www2.census.gov/econ/bps/Place/{region}%20Region/{prefix}{year}a.txt'
BPS_REGIONS = {'West': 'we', 'Midwest': 'mw', 'Northeast': 'ne', 'South': 'so'}
PERMIT_YEARS = list(range(2010, 2026))
BPS_SIZES = {'u1': '1-unit', 'u2': '2-units', 'u34': '3-4 units', 'u5': '5+ units'}

# Seventeen identifier columns, then buildings/units/value for each of the four structure sizes,
# then the same four repeated as reported-only. The header spans two lines and is skipped.
BPS_COLUMNS = ['survey_year', 'state', 'id6', 'county', 'census_place', 'place', 'mcd', 'pop',
               'csa', 'cbsa', 'footnote', 'central_city', 'zip', 'region', 'division',
               'months_reported', 'place_name']
for prefix in ['u1', 'u2', 'u34', 'u5', 'r1', 'r2', 'r34', 'r5']:
    BPS_COLUMNS += [f'{prefix}_bldgs', f'{prefix}_units', f'{prefix}_value']

def read_bps_file(region, prefix, year):
    response = requests.get(BPS_URL.format(region=region, prefix=prefix, year=year), timeout=120)
    response.raise_for_status()
    body = '\n'.join(response.text.split('\n')[3:])       # two header lines and a blank
    frame = pd.read_csv(io.StringIO(body), header=None, names=BPS_COLUMNS, dtype=str,
                        usecols=range(len(BPS_COLUMNS)), on_bad_lines='skip')
    frame['place_fips'] = (frame['state'].str.strip().str.zfill(2)
                           + frame['place'].str.strip().str.zfill(5))
    return frame[frame['place_fips'].isin(set(places_df['place_fips']))].assign(year=year)

bps_raw_df = pd.concat([read_bps_file(region, prefix, year)
                        for year in PERMIT_YEARS for region, prefix in BPS_REGIONS.items()],
                       ignore_index=True)

# Tidy: one row per place, year and structure size
place_permit_l = []
for key, label in BPS_SIZES.items():
    part_df = bps_raw_df[['place_fips', 'year', 'months_reported', f'{key}_bldgs', f'{key}_units']].copy()
    part_df.columns = ['place_fips', 'year', 'months_reported', 'buildings', 'units']
    place_permit_l.append(part_df.assign(size_class=label))
place_permits_df = pd.concat(place_permit_l, ignore_index=True)
for column in ['months_reported', 'buildings', 'units']:
    place_permits_df[column] = pd.to_numeric(place_permits_df[column], errors='coerce')
place_permits_df = (place_permits_df[['place_fips', 'year', 'size_class', 'months_reported',
                                      'buildings', 'units']]
                    .sort_values(['place_fips', 'year', 'size_class']).reset_index(drop=True))

# Integrity checks
bps_places_s = set(place_permits_df['place_fips'])
absent_l = sorted(set(places_df['place_fips']) - bps_places_s)
assert place_permits_df['units'].notna().all(), 'a null unit count survived'
assert place_permits_df['units'].ge(0).all(), 'a negative unit count'
years_per_place_s = place_permits_df.groupby('place_fips')['year'].nunique()
assert (years_per_place_s == len(PERMIT_YEARS)).all(), 'a place is missing a year'
assert len(place_permits_df) == len(bps_places_s) * len(PERMIT_YEARS) * len(BPS_SIZES), 'row count is wrong'

partial_n = int((place_permits_df.drop_duplicates(['place_fips', 'year'])['months_reported'] < 12).sum())
print(f'{len(bps_places_s)} of {len(places_df)} registry places file with the BPS, '
      f'{PERMIT_YEARS[0]}-{PERMIT_YEARS[-1]}')
print(f"  no separate place record: {', '.join(places_df.set_index('place_fips').loc[absent_l, 'name']) or 'none'}")
print(f'  place-years reporting fewer than 12 months (the Census imputes the rest): {partial_n}')
bps_raw_path = RAW / 'bps' / 'place-permits-raw.csv'
bps_raw_path.parent.mkdir(parents=True, exist_ok=True)
bps_raw_df.to_csv(bps_raw_path, index=False)
written_l.append({'file': str(bps_raw_path), 'rows': len(bps_raw_df), 'cols': bps_raw_df.shape[1],
                  'note': 'BPS rows for the registry places, every column as supplied'})
print(f'{str(bps_raw_path):<48} {len(bps_raw_df):>7,} rows x {bps_raw_df.shape[1]} cols')
write_table(place_permits_df, 'place-permits.csv', 'units by place, year and structure size')
place_permits_df.head()

30 of 31 registry places file with the BPS, 2010-2025
  no separate place record: Bloomington
  place-years reporting fewer than 12 months (the Census imputes the rest): 97
data/raw/bps/place-permits-raw.csv                   480 rows x 43 cols
data/processed/place-permits.csv                   1,920 rows x 6 cols   units by place, year and structure size


,place_fips,year,size_class,months_reported,buildings,units
0,0137000,2010,1-unit,12,1073,1073
1,0137000,2010,2-units,12,0,0
2,0137000,2010,3-4 units,12,0,0
3,0137000,2010,5+ units,12,0,0
4,0137000,2011,1-unit,12,1018,1018


In [11]:
# County totals from FRED, for the 26 basket counties plus Broomfield and the Front Range
FRED_PERMIT_COUNTIES = sorted(SEDAC_KEEP | {fips for fips in sya_df.loc[
    sya_df['county'].isin(FRONT_RANGE_COUNTIES), 'countyfips'].map(lambda code: f'08{code:03d}')})

def permit_series_id(county_fips):
    # FRED pads the county FIPS to six digits: Boulder 08013 becomes BPPRIV008013
    return 'BPPRIV' + str(int(county_fips)).zfill(6)

county_permit_raw_d, county_permit_l = {}, []
for county_fips in FRED_PERMIT_COUNTIES:
    series_id = permit_series_id(county_fips)
    observations = fred_observations(series_id)
    county_permit_raw_d[series_id] = observations
    county_permit_l.extend({'county_fips': county_fips, 'series_id': series_id,
                            'year': int(row['date'][:4]), 'units': float(row['value'])}
                           for row in observations if row['value'] != '.')
county_permits_df = (pd.DataFrame(county_permit_l)
                     .sort_values(['county_fips', 'year']).reset_index(drop=True))

# Integrity checks
assert set(county_permits_df['county_fips']) == set(FRED_PERMIT_COUNTIES), 'a county is missing'
assert county_permits_df['units'].ge(0).all(), 'a negative county unit count'
assert not county_permits_df.duplicated(['county_fips', 'year']).any(), 'duplicate county-year'
short_l = [fips for fips, frame in county_permits_df.groupby('county_fips')
           if frame['year'].max() < PERMIT_YEARS[-1]]
print(f'{len(FRED_PERMIT_COUNTIES)} counties · {county_permits_df.year.min()}-{county_permits_df.year.max()}')
print(f"  series ending before {PERMIT_YEARS[-1]}: "
      f"{ {fips: int(county_permits_df.loc[county_permits_df.county_fips == fips, 'year'].max()) for fips in short_l} or 'none'}")
write_raw(county_permit_raw_d, 'fred/permit-observations.json', f'{len(county_permit_raw_d)} BPPRIV series')
write_table(county_permits_df, 'county-permits.csv', 'FRED BPPRIV, annual units by county')
county_permits_df.head()

41 counties · 1990-2025
  series ending before 2025: {'09009': 2022}
data/raw/fred/permit-observations.json           177,836 bytes   41 BPPRIV series
data/processed/county-permits.csv                  1,460 rows x 4 cols   FRED BPPRIV, annual units by county


,county_fips,series_id,year,units
0,01089,BPPRIV001089,1990,1865.0
1,01089,BPPRIV001089,1991,1196.0
2,01089,BPPRIV001089,1992,1778.0
3,01089,BPPRIV001089,1993,1899.0
4,01089,BPPRIV001089,1994,850.0


## The comprehensive plan corpus

Fifteen editions of the Boulder Valley Comprehensive Plan sit in `data/raw/bvcp/` as Markdown
conversions. The cell below strips the Markdown syntax, tables, page numbers and conversion
artifacts, then counts how often each term of three keyword families appears, per edition.

The counting is mechanical and belongs here. The choice of terms is not, and the analysis notebook
defends it and tests it. So this cell writes one row per **term** per edition, not one row per
family: from term counts the analysis can add up a family, and can drop any one term and see where
the finding moves. The word count of each edition rides along, because every rate the column
reports is per 1,000 words.

Two matching rules are worth stating here, because they are in the patterns:

- `famil*` is matched only where it is **not** preceded by single, multi, multiple or two. Without
  that exclusion, 180 of its 343 occurrences are "single-family" and similar, which is zoning
  vocabulary rather than attention to families.
- Character is matched as a phrase — neighborhood, community, historic or rural character — rather
  than as a stem. The bare stem picks up 271 occurrences of "characteristics" and "characterized".

In [12]:
BVCP_DIR = RAW / 'bvcp'
BVCP_EDITIONS = {
    'bvcp-1977-aug.md': ('1977', 1977), 'bvcp-1978-jun.md': ('1978', 1978),
    'bvcp-1979-jun.md': ('1979', 1979), 'bvcp-1981-may.md': ('1981', 1981),
    'bvcp-1983-oct.md': ('1983', 1983), 'bvcp-1986-oct.md': ('1986', 1986),
    'bvcp-1990-dec.md': ('1990', 1990), 'bvcp-1996-nov.md': ('1996', 1996),
    'bvcp-2001-sep.md': ('2001', 2001), 'bvcp-2005-dec.md': ('2005', 2005),
    'bvcp-2008.md': ('2008', 2008), 'bvcp-2010.md': ('2010', 2010),
    'bvcp-2015.md': ('2015', 2015), 'bvcp-2021-mar.md': ('2021', 2021),
    'bvcp-2026-mar.md': ('2026 draft', 2026),
}

# 'family' only where it is not part of single-family, multi-family and the like
FAMILY_NOT_HOUSING_TYPE = (r'(?<!single-)(?<!single )(?<!multi-)(?<!multi )'
                           r'(?<!multiple-)(?<!multiple )(?<!two-)(?<!two )\bfamil[a-z]*')

KEYWORD_FAMILIES = {
    'Preservation': [r'\bpreserv[a-z]*', r'\bopen\s+space', r'\bneighborhood character',
                     r'\bcommunity character', r'\bhistoric character', r'\brural character',
                     r'\bgrowth management', r'\bcompatib[a-z]*'],
    'Children & schools': [FAMILY_NOT_HOUSING_TYPE, r'\bchild[a-z]*', r'\bschool[a-z]*',
                           r'\bstudent[a-z]*', r'\byouth[a-z]*', r'\bplayground[a-z]*'],
    'Housing affordability': [r'\bafford[a-z]*'],
}

def clean_text(md_text):
    # Strip markdown syntax, tables, page numbers and conversion artifacts
    text = re.sub(r'!\[.*?\]\(.*?\)', '', md_text)
    text = re.sub(r'\[([^\]]*)\]\([^\)]*\)', r'\1', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'^#{1,6}\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'\*{1,3}([^*]+)\*{1,3}', r'\1', text)
    text = re.sub(r'\|', ' ', text)
    text = re.sub(r'^[-:]+$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\.{3,}\s*\d+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

corpus_d = {}
for filename, (label, year) in sorted(BVCP_EDITIONS.items(), key=lambda kv: kv[1][1]):
    text = clean_text((BVCP_DIR / filename).read_text(encoding='utf-8', errors='replace'))
    corpus_d[label] = {'year': year, 'clean': text.lower(), 'word_count': len(text.split())}

# Terms are matched as phrases and word-prefixes over the cleaned text, so a stem or a
# multi-word term is visible. One row per edition x term.
term_records_l = []
for label, info in corpus_d.items():
    for family, terms in KEYWORD_FAMILIES.items():
        for term in terms:
            term_records_l.append({'version': label, 'year': info['year'], 'family': family,
                                   'term': term, 'count': len(re.findall(term, info['clean'])),
                                   'word_count': info['word_count']})
bvcp_terms_df = pd.DataFrame(term_records_l)

# Integrity checks
assert len(corpus_d) == 15, f'expected 15 editions; got {len(corpus_d)}'
assert all(info['word_count'] > 10_000 for info in corpus_d.values()), 'an edition converted to near-empty text'
assert (bvcp_terms_df.groupby('family')['count'].sum() > 0).all(), 'a family scored zero'
assert len(bvcp_terms_df) == len(corpus_d) * sum(len(t) for t in KEYWORD_FAMILIES.values())

print(f"{len(corpus_d)} editions, {sum(i['word_count'] for i in corpus_d.values()):,} words total")
print(bvcp_terms_df.groupby('family')['count'].sum().to_string())
write_table(bvcp_terms_df, 'bvcp-term-counts.csv', 'one row per edition x term')
bvcp_terms_df.head()

15 editions, 707,475 words total
family
Children & schools       1095
Housing affordability     434
Preservation             3092
data/processed/bvcp-term-counts.csv                  225 rows x 6 cols   one row per edition x term


,version,year,family,term,count,word_count
0,1977,1977,Preservation,\bpreserv[a-z]*,32,22073
1,1977,1977,Preservation,\bopen\s+space,55,22073
2,1977,1977,Preservation,\bneighborhood character,1,22073
3,1977,1977,Preservation,\bcommunity character,0,22073
4,1977,1977,Preservation,\bhistoric character,1,22073


## Open enrollment matrices

BVSD publishes one open enrollment matrix per school level per year as a PDF: rows are the school
attended, columns the neighborhood attendance area the student lives in, cells the number of
students. The section below fetches any file it does not already hold, parses all 31, and writes a
tidy edgelist with three summary tables.

These tables are not read by `bvsd-analysis.ipynb`. The column does not use open enrollment. They
serve the archived notebooks in `appendix/`, and they are built here because this is where
raw-to-processed work lives.

The parser reports every reconciliation failure into an audit report rather than stopping, because
a single bad cell in one year should not cost the other thirty files. Read
`data/processed/open-enrollment/audit_report.md` after a run.

In [13]:
OE_RAW = RAW / 'open-enrollment'
OE_OUT = PROC / 'open-enrollment'
OE_PAGE_URL = ('https://www.bvsd.org/departments/operational-services/planning-and-engineering')
OE_LEVELS = {'elem': 'elementary', 'middle': 'middle', 'high': 'high'}
PSEUDO_OUTSIDE, PSEUDO_UNMATCHED, PSEUDO_PLACEMENT = 'OUTSIDE_DISTRICT', 'UNMATCHED_ADDRESS', 'PLACEMENT'

SUMMARY_COLS = [   # fixed positional order of the right block
    'attending_neighborhood', 'oe_from_within_district', 'oe_from_outside_district',
    'placements_into_school', 'unmatched_addresses', 'within_bvsd_total', 'enrollment_total',
    'pct_from_neighborhood']
AREA_ROWS = [      # fixed positional order of the bottom block
    'placed_out', 'living_in_area_total', 'oe_out_of_area', 'pct_enrolled_in_neighborhood']

def oe_discover_links():
    html = subprocess.run(['curl', '-sL', OE_PAGE_URL], capture_output=True,
                          text=True, check=True).stdout
    links = re.findall(r'href="(https://resources\.finalsite\.net/[^"]*Matrix[^"]*\.pdf)"', html)
    rows = []
    for url in dict.fromkeys(links):        # de-duplicate, keep order
        filename = url.rsplit('/', 1)[1]
        match = re.search(r'(20\d\d)-(\d\d)', filename)
        key = next((k for k in OE_LEVELS if filename.lower().startswith(k)), None)
        if not match or key is None:        # K / 6th / 9th entry-grade files are skipped
            continue
        first_year = int(match.group(1))
        rows.append(dict(level=OE_LEVELS[key], school_year=f'{first_year}-{first_year + 1}',
                         url=url, source_filename=filename))
    return rows

def oe_fetch(refetch=False):
    OE_RAW.mkdir(parents=True, exist_ok=True)
    manifest_path = OE_RAW / 'manifest.json'
    if manifest_path.exists() and not refetch:
        manifest_l = json.loads(manifest_path.read_text())
        if all((OE_RAW / row['file']).exists() for row in manifest_l):
            return manifest_l
    manifest_l = []
    for row in oe_discover_links():
        target = OE_RAW / f"{row['level']}_{row['school_year']}.pdf"
        if refetch or not target.exists():
            subprocess.run(['curl', '-sL', '-o', str(target), row['url']], check=True)
        manifest_l.append(dict(row, file=target.name, bytes=target.stat().st_size,
                               sha256=hashlib.sha256(target.read_bytes()).hexdigest()))
    manifest_l.sort(key=lambda row: (row['school_year'], row['level']))
    manifest_path.write_text(json.dumps(manifest_l, indent=1))
    return manifest_l

oe_manifest_l = oe_fetch()
print(f'{len(oe_manifest_l)} matrices · '
      f"{sorted({row['school_year'] for row in oe_manifest_l})[0]} to "
      f"{sorted({row['school_year'] for row in oe_manifest_l})[-1]}")

30 matrices · 2016-2017 to 2025-2026


In [14]:
def _decode_rotated(page, bbox):
    # Rebuild the text of a rotated header cell from character positions. pdfplumber reads
    # rotated glyphs in stream order, which scrambles them; sorting by column of glyphs and
    # then vertical position restores reading order.
    x0, top, x1, bottom = bbox
    chars = [c for c in page.chars
             if c['x0'] >= x0 - 1 and c['x1'] <= x1 + 1
             and c['top'] >= top - 1 and c['bottom'] <= bottom + 1]
    if not chars:
        return ''
    if chars[0]['matrix'][1] > 0:      # rotated counter-clockwise: reads bottom to top
        chars.sort(key=lambda c: (round(c['x0'] / 3), -c['top']))
    else:                              # rotated clockwise: reads top to bottom
        chars.sort(key=lambda c: (round(-c['x0'] / 3), c['top']))
    return re.sub(r'\s+', ' ', ''.join(c['text'] for c in chars)).strip()

def _num(value):
    # '1,234' -> 1234; '' or None -> None; strips stray annotation text
    if value is None:
        return None
    text = str(value).split('\n')[0].strip().replace(',', '')
    match = re.match(r'^-?\d+$', text)
    return int(match.group()) if match else None

def _pct(value):
    if value is None:
        return None
    match = re.search(r'(\d+(?:\.\d+)?)\s*%', str(value))
    return float(match.group(1)) / 100 if match else None

def _clean(value):
    return re.sub(r'\s+', ' ', (value or '')).strip()

def parse_pdf(path, errors_l):
    # Returns the matrix cells, the school summaries, the area summaries, the district totals
    # and the header metadata of one file.
    rec = dict(file=path.name)
    with pdfplumber.open(path) as pdf:
        page = pdf.pages[0]
        if len(pdf.pages) > 1:
            errors_l.append((path.name, 'structure', f'{len(pdf.pages)} pages; only page 1 parsed'))
        lines = (page.extract_text() or '').split('\n')
        match = re.search(r'MATRIX,\s*(\d{4}-\d{4})\s*(\d{1,2}/\d{1,2}/\d{2,4})?',
                          lines[0] if lines else '')
        rec['title_year'] = match.group(1) if match else None
        rec['matrix_date'] = match.group(2) if match else None
        rec['grade_line'] = _clean(lines[1]) if len(lines) > 1 else None
        rec['footnotes'] = ' | '.join(line for line in lines if line.startswith('*'))

        tables = page.find_tables()
        if not tables:
            errors_l.append((path.name, 'structure', 'no table found'))
            return rec
        table = tables[0]
        grid = table.extract()
        headers = [_decode_rotated(page, cell) if cell else '' for cell in table.rows[0].cells]

    # Locate the column blocks. Column 0 is a row-group marker, column 1 the school name, then the
    # area columns, then a blank-header column repeating the school name, then 8 summary columns.
    ncol = len(grid[0])
    blank_after = [i for i in range(2, ncol) if headers[i] == '' and headers[i - 1]]
    if not blank_after:
        errors_l.append((path.name, 'structure', 'cannot find summary block'))
        return rec
    name_repeat_col = blank_after[0]
    area_cols = list(range(2, name_repeat_col))
    summary_cols = list(range(name_repeat_col + 1, ncol))
    if len(summary_cols) != 8:
        errors_l.append((path.name, 'structure', f'expected 8 summary columns, found '
                                                 f'{len(summary_cols)}: {[headers[i] for i in summary_cols]}'))
    summary_hdr = ' '.join(headers[i] for i in summary_cols).lower()
    for keyword in ['neighborhood', 'within', 'outside', 'placement', 'unmatched', 'total', '%']:
        if keyword not in summary_hdr:
            errors_l.append((path.name, 'header', f"summary header keyword '{keyword}' not found"))
    areas = [_clean(headers[i]) for i in area_cols]
    if any(area == '' for area in areas):
        errors_l.append((path.name, 'header', f'blank area header in {areas}'))

    # Locate the row blocks
    body = grid[1:]
    first_summary_row = next((i for i, row in enumerate(body)
                              if row[0] and re.search(r'placed', row[0], re.I)), None)
    if first_summary_row is None:
        errors_l.append((path.name, 'structure', 'cannot find bottom block'))
        return rec
    school_rows = [row for row in body[:first_summary_row] if _clean(row[1])]
    area_rows = body[first_summary_row:first_summary_row + 4]
    if len(area_rows) != 4:
        errors_l.append((path.name, 'structure', f'expected 4 bottom rows, found {len(area_rows)}'))

    # Matrix cells and school summaries
    matrix, school_summary = [], []
    for row in school_rows:
        school = _clean(row[1])
        if not school:
            continue
        for j, area in zip(area_cols, areas):
            students = _num(row[j])
            if students:
                matrix.append((school, area, students))
        summary = dict(school_raw=school,
                       matrix_row_sum=sum(_num(row[j]) or 0 for j in area_cols))
        for key, j in zip(SUMMARY_COLS, summary_cols):
            summary[key] = _pct(row[j]) if key.startswith('pct') else _num(row[j])
        repeated = _clean(row[name_repeat_col])
        if repeated and repeated.replace(' ', '') != school.replace(' ', ''):
            errors_l.append((path.name, 'header',
                             f'repeated school name mismatch: {school!r} vs {repeated!r}'))
        school_summary.append(summary)

    # Area summaries and district totals
    area_summary = {area: dict(area_raw=area) for area in areas}
    for key, row in zip(AREA_ROWS, area_rows):
        for j, area in zip(area_cols, areas):
            area_summary[area][key] = _pct(row[j]) if key.startswith('pct') else _num(row[j])
    for j, area in zip(area_cols, areas):
        area_summary[area]['matrix_col_sum'] = sum(_num(row[j]) or 0 for row in school_rows)
    district = {}
    for key, j in zip(SUMMARY_COLS, summary_cols):
        value = area_rows[0][j] if j < len(area_rows[0]) else None
        district[key] = _pct(value) if key.startswith('pct') else _num(value)
    for key, j in zip(SUMMARY_COLS[:4], summary_cols[:4]):
        district[key + '_share'] = _pct(area_rows[1][j]) if len(area_rows) > 1 else None

    rec.update(areas=areas, schools=[s['school_raw'] for s in school_summary], matrix=matrix,
               school_summary=school_summary, area_summary=list(area_summary.values()),
               district=district)
    return rec

def canonical(name):
    # One spelling per school and per area, across nine years of shifting abbreviations
    text = re.sub(r'\s+', ' ', name.replace('*', '')).strip()
    text = re.sub(r'\s*/\s*', '/', text)
    text = re.sub(r'\b(Elem\.|Elem|Elementary)$', '', text).strip()
    text = re.sub(r'\bInt\.$', 'International', text)
    text = re.sub(r'^Comm\. ', 'Community ', text)
    text = re.sub(r'^Uni-Hill', 'University Hill', text)
    text = text.replace('Peak-to-Peak', 'Peak to Peak')
    text = re.sub(r'^Contract(ed)? Ed( Prog)?$', 'Contracted Ed Program', text)
    text = text.replace('Middle School', 'Middle').replace('Middle Sch.', 'Middle')
    text = re.sub(r'\bK-?8\b', 'K-8', text)
    text = re.sub(r'\bPK-8\b', 'K-8', text)
    return text.strip()

print('parser ready')

parser ready


In [15]:
def audit(recs_l, errors_l):
    # Every published total must reconcile with the cells it is made of. Failures are recorded,
    # never raised: one bad cell in one year must not cost the other thirty files.
    for rec in recs_l:
        name = rec['file']
        if 'matrix' not in rec:
            continue
        for summary in rec['school_summary']:
            expected = (summary['attending_neighborhood'] or 0) + (summary['oe_from_within_district'] or 0)
            if summary['matrix_row_sum'] != expected:
                errors_l.append((name, 'row_sum', f"{summary['school_raw']}: matrix row sum "
                                                  f"{summary['matrix_row_sum']} != attending+oe_within {expected}"))
            parts = [summary[key] or 0 for key in SUMMARY_COLS[:5]]
            if summary['enrollment_total'] is not None and sum(parts) != summary['enrollment_total']:
                errors_l.append((name, 'school_total', f"{summary['school_raw']}: sum of 5 components "
                                                       f"{sum(parts)} != enrollment_total {summary['enrollment_total']}"))
            inside = sum(parts) - (summary['oe_from_outside_district'] or 0)
            if summary['within_bvsd_total'] is not None and inside != summary['within_bvsd_total']:
                errors_l.append((name, 'within_total', f"{summary['school_raw']}: components minus outside "
                                                       f"{inside} != within_bvsd_total {summary['within_bvsd_total']}"))
        for area in rec['area_summary']:
            expected = area['matrix_col_sum'] + (area['placed_out'] or 0)
            if area['living_in_area_total'] is not None and expected != area['living_in_area_total']:
                errors_l.append((name, 'col_sum', f"{area['area_raw']}: matrix column sum "
                                                  f"{area['matrix_col_sum']} + placed_out {area['placed_out']} "
                                                  f"!= living_in_area_total {area['living_in_area_total']}"))
        for key in SUMMARY_COLS[:7]:
            total = sum(summary[key] or 0 for summary in rec['school_summary'])
            if rec['district'].get(key) is not None and total != rec['district'][key]:
                errors_l.append((name, 'district_total',
                                 f"{key}: sum of schools {total} != district {rec['district'][key]}"))
        # The diagonal, plus any optional area that names the school, must equal attending_neighborhood
        cells_d = defaultdict(int)
        for school, area, students in rec['matrix']:
            cells_d[(canonical(school), canonical(area))] += students
        area_canon_l = [canonical(area) for area in rec['areas']]
        for summary in rec['school_summary']:
            school_canon = canonical(summary['school_raw'])
            if school_canon not in area_canon_l:
                continue
            own = [a for a in area_canon_l
                   if a == school_canon or ('Optional' in a and re.search(rf'\b{re.escape(school_canon)}\b', a))]
            got = sum(cells_d.get((school_canon, a), 0) for a in own)
            if got != (summary['attending_neighborhood'] or 0):
                errors_l.append((name, 'diagonal', f"{summary['school_raw']}: own-area cells {got} "
                                                   f"({'+'.join(own)}) != attending_neighborhood "
                                                   f"{summary['attending_neighborhood']}"))
        if rec.get('title_year') and rec['title_year'] != name.split('_')[1].replace('.pdf', ''):
            errors_l.append((name, 'provenance', f"title year {rec['title_year']} != filename year"))

OE_OUT.mkdir(parents=True, exist_ok=True)
oe_errors_l, oe_recs_l = [], []
for entry in oe_manifest_l:
    rec = parse_pdf(OE_RAW / entry['file'], oe_errors_l)
    rec.update(level=entry['level'], school_year=entry['school_year'], url=entry['url'],
               sha256=entry['sha256'])
    oe_recs_l.append(rec)
audit(oe_recs_l, oe_errors_l)
print(f"parsed {sum('matrix' in rec for rec in oe_recs_l)} of {len(oe_recs_l)} files · "
      f'{len(oe_errors_l)} reconciliation issues')

parsed 30 of 30 files · 2 reconciliation issues


In [16]:
# Crosswalk: raw name -> canonical name, per level
names_s = set()
for rec in oe_recs_l:
    names_s |= {(rec['level'], name, 'school') for name in rec.get('schools', [])}
    names_s |= {(rec['level'], name, 'area') for name in rec.get('areas', [])}
roles_d = defaultdict(set)
for level, name, role in names_s:
    roles_d[(level, name)].add(role)
crosswalk_df = (pd.DataFrame([dict(level=level, name_raw=name, canonical=canonical(name),
                                   appears_as='+'.join(sorted(roles)))
                              for (level, name), roles in roles_d.items()])
                .sort_values(['level', 'canonical', 'name_raw']).reset_index(drop=True))
canon_d = {(level, name): canon for level, name, canon
           in crosswalk_df[['level', 'name_raw', 'canonical']].itertuples(index=False)}

edges_l = []
for rec in oe_recs_l:
    if 'matrix' not in rec:
        continue
    base = dict(school_year=rec['school_year'], level=rec['level'])
    for school, area, students in rec['matrix']:
        edges_l.append(dict(base, source_raw=area, target_raw=school,
                            source=canon_d[(rec['level'], area)],
                            target=canon_d[(rec['level'], school)],
                            students=students, edge_type='matrix'))
    for summary in rec['school_summary']:
        target = canon_d[(rec['level'], summary['school_raw'])]
        for column, pseudo in [('oe_from_outside_district', PSEUDO_OUTSIDE),
                               ('unmatched_addresses', PSEUDO_UNMATCHED),
                               ('placements_into_school', PSEUDO_PLACEMENT)]:
            if summary[column]:
                edges_l.append(dict(base, source_raw=pseudo, target_raw=summary['school_raw'],
                                    source=pseudo, target=target, students=summary[column],
                                    edge_type=column))
edgelist_df = pd.DataFrame(edges_l)
edgelist_df['self_loop'] = edgelist_df['source'] == edgelist_df['target']
edgelist_df = edgelist_df[['school_year', 'level', 'source', 'target', 'students', 'self_loop',
                           'edge_type', 'source_raw', 'target_raw']]

school_summary_df = pd.DataFrame([dict(school_year=rec['school_year'], level=rec['level'],
                                       school=canon_d[(rec['level'], summary['school_raw'])], **summary)
                                  for rec in oe_recs_l if 'matrix' in rec
                                  for summary in rec['school_summary']])
area_summary_df = pd.DataFrame([dict(school_year=rec['school_year'], level=rec['level'],
                                     area=canon_d[(rec['level'], area['area_raw'])], **area)
                                for rec in oe_recs_l if 'matrix' in rec
                                for area in rec['area_summary']])
district_summary_df = pd.DataFrame([dict(school_year=rec['school_year'], level=rec['level'],
                                         **rec['district'])
                                    for rec in oe_recs_l if 'matrix' in rec])
provenance_df = pd.DataFrame([dict(school_year=rec['school_year'], level=rec['level'],
                                   file=rec['file'], url=rec['url'], sha256=rec['sha256'],
                                   matrix_date=rec.get('matrix_date'), grade_line=rec.get('grade_line'),
                                   footnotes=rec.get('footnotes'), n_schools=len(rec.get('schools', [])),
                                   n_areas=len(rec.get('areas', [])), n_edges=len(rec.get('matrix', [])))
                              for rec in oe_recs_l])

# Integrity checks
assert edgelist_df['students'].gt(0).all(), 'a zero or negative edge survived'
assert set(provenance_df['school_year']) == {row['school_year'] for row in oe_manifest_l}
assert len(provenance_df) == len(oe_manifest_l), 'a file is missing from provenance'

for frame, name in [(edgelist_df, 'edgelist.csv'), (school_summary_df, 'school_summary.csv'),
                    (area_summary_df, 'area_summary.csv'), (district_summary_df, 'district_summary.csv'),
                    (crosswalk_df, 'crosswalk.csv'), (provenance_df, 'provenance.csv')]:
    write_table(frame, f'open-enrollment/{name}')

report_l = ['# Audit report', '',
            f"Files parsed: {sum('matrix' in rec for rec in oe_recs_l)} / {len(oe_recs_l)}",
            f"Edges (matrix): {int((edgelist_df.edge_type == 'matrix').sum())}",
            f"Edges (pseudo-source): {int((edgelist_df.edge_type != 'matrix').sum())}",
            f'Total students in edgelist: {int(edgelist_df.students.sum())}',
            f'Issues found: {len(oe_errors_l)}', '']
if oe_errors_l:
    report_l += ['| file | check | detail |', '|---|---|---|']
    report_l += [f'| {f} | {c} | {d} |' for f, c, d in oe_errors_l]
else:
    report_l.append('All checks passed.')
report_l += ['', '## Per-file counts', '',
             provenance_df[['school_year', 'level', 'matrix_date', 'n_schools', 'n_areas',
                            'n_edges']].to_markdown(index=False)]
(OE_OUT / 'audit_report.md').write_text('\n'.join(report_l))
written_l.append({'file': str(OE_OUT / 'audit_report.md'), 'rows': np.nan, 'cols': np.nan,
                  'note': f'{len(oe_errors_l)} issues'})
print('\n'.join(report_l[:7]))

data/processed/open-enrollment/edgelist.csv        7,999 rows x 9 cols   
data/processed/open-enrollment/school_summary.csv     690 rows x 13 cols   
data/processed/open-enrollment/area_summary.csv      498 rows x 9 cols   
data/processed/open-enrollment/district_summary.csv      30 rows x 14 cols   
data/processed/open-enrollment/crosswalk.csv          85 rows x 4 cols   
data/processed/open-enrollment/provenance.csv         30 rows x 11 cols   
# Audit report

Files parsed: 30 / 30
Edges (matrix): 6605
Edges (pseudo-source): 1394
Total students in edgelist: 285601
Issues found: 2


## Manifest

Everything this notebook wrote. `bvsd-analysis.ipynb` reads the `data/processed/` rows and nothing
else; the `data/raw/` rows are the API answers kept as returned, so the tidying above can be
checked against its source without a key and without a new pull.

In [17]:
manifest_df = pd.DataFrame(written_l)
manifest_df['kind'] = np.where(manifest_df['file'].str.startswith('data/raw'), 'raw', 'processed')

assert manifest_df['file'].is_unique, 'a file was written twice'
assert all(Path(path).exists() for path in manifest_df['file']), 'a listed file is not on disk'
assert (manifest_df['kind'] == 'processed').sum() >= 18, 'fewer processed tables than expected'

print(f"{len(manifest_df)} files · {(manifest_df.kind == 'processed').sum()} processed, "
      f"{(manifest_df.kind == 'raw').sum()} raw")
print(f"{int(manifest_df['rows'].sum()):,} rows written in total")
manifest_df[['kind', 'file', 'rows', 'cols', 'note']]

29 files · 23 processed, 6 raw
192,504 rows written in total


,kind,file,rows,cols,note
0,processed,data/processed/places.csv,31.0,6.0,"the registry, validated"
1,processed,data/processed/place-age-groups.csv,2356.0,6.0,"31 places x [1990, 2000, 2010, 2020]"
2,processed,data/processed/county-age-groups.csv,2090.0,6.0,28 counties (Broomfield exists from 2010)
3,processed,data/processed/place-age-single.csv,1860.0,4.0,"ages 0-19 · [2000, 2010, 2020]"
4,processed,data/processed/sdo-sya-frontrange.csv,125478.0,5.0,17 Front Range counties + the Colorado row
5,processed,data/processed/sdo-components-bvsd.csv,152.0,5.0,"births, deaths, net migration"
6,processed,data/processed/sdo-households-bvsd.csv,2050.0,5.0,households by type and householder age
7,processed,data/processed/sedac-county-age-ssp.csv,41310.0,7.0,"basket counties, all five SSPs"
8,processed,data/processed/sedac-national-age-ssp.csv,1530.0,6.0,every US county summed
9,raw,data/raw/census/national-p12-2010.json,NaN,NaN,"table P12, 2010, as returned"


### What this notebook does not do

- It does not fill a gap. Where a source publishes no value — income for 1990 to 1996, New Haven
  after 2021, Broomfield County before 2010 — the cell is absent from the tidy table, not
  interpolated.
- It does not reconcile two sources to each other. The SEDAC 2020 projection and the 2020 census
  count of the same county differ, and both are written as published.
- It does not compute a measure the column reports. No percent change, no ratio, no projection.
- It does not redistribute a published figure across a finer grain. Every age bucket the analysis
  uses is an exact sum of rows a census published.
- It does not correct a source. The open enrollment audit reports the reconciliation failures it
  finds and writes the numbers as published.